# Baseline RAG — SEC Filings

**Pipeline**: Query → Dense Retrieval (top-5) → LLM Generation

The simplest possible RAG baseline:
- Embed the raw query with `all-mpnet-base-v2`
- Retrieve top-5 chunks from the pre-built ChromaDB vector store
- Pass context + question to `llama-3.1-8b-instant` (dev) or `llama-4-scout-17b` (final)
- No query transformation, no metadata filtering, no reranking

Results saved to `./results/baseline_results.csv` for comparison with Advanced RAG.

In [1]:
print('hi')

hi


In [2]:
# Uncomment to install dependencies
# !pip install sentence-transformers chromadb groq python-dotenv pandas tqdm

## 1. Configuration

In [3]:
CONFIG = {
    # --- Paths (relative to this notebook) ---
    "chroma_db_path": "../sec_rag_team_share/chroma_db",
    "chunks_path":    "../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl",
    "eval_path":      "../sec_rag_team_share/evaluation/GenAI Eval QA.csv",
    "results_dir":    "./results",

    # --- Models ---
    "dense_model_name":   "sentence-transformers/all-mpnet-base-v2",
    # Execution profile: 'dev' or 'final' (same model — kept for parity with Advanced RAG)
    "execution_profile":  "dev",
    "gemini_dev_model":   "gemini-2.5-flash",
    "gemini_final_model": "gemini-2.5-flash",
    "judge_model":        "gemini-2.5-flash",

    # --- Retrieval ---
    "dense_top_k":       5,
    "max_context_chars": 7000,

    # --- Generation ---
    "temperature_generator": 0.2,
    "temperature_judge":     0.1,   # 0.0 causes empty responses in Gemini via LiteLLM
    "max_tokens_answer":     512,
    "max_tokens_judge":      256,

    # --- Cost tracking (gemini-2.5-flash-lite) ---
    "gemini_cost_input_per_1m":  0.10,
    "gemini_cost_output_per_1m": 0.40,

    # --- Evaluation ---
    "eval_split": "test",

    # --- Rate limiting (Gemini free tier: 10 RPM) ---
    # Baseline makes 3 LLM calls per question: generate + judge_corr + judge_faith
    "max_rpm":                    10,
    "inter_question_sleep_sec":   20,
    "llm_max_retries":            3,
    "llm_retry_base_delay_sec":   5,
}

## 2. Imports & Setup

In [4]:
import os
import json
import time
import threading
from collections import deque, Counter as _Counter
from pathlib import Path
from typing import Optional
import re as _re

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types as genai_types

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your .env file"

LLM_MODEL = (
    CONFIG["gemini_final_model"]
    if CONFIG["execution_profile"] == "final"
    else CONFIG["gemini_dev_model"]
)
print(f"Execution profile : {CONFIG['execution_profile']}")
print(f"LLM model         : {LLM_MODEL}")

c:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Execution profile : dev
LLM model         : gemini-2.5-flash


## 3. Load Models

In [5]:
print("Loading embedding model...")
embed_model = SentenceTransformer(CONFIG["dense_model_name"])
print(f"  ok {CONFIG['dense_model_name']}")

genai_client = genai.Client(api_key=GEMINI_API_KEY)
print(f"  ok Gemini client ready (model: {LLM_MODEL})")


# ── Rate Limiter ───────────────────────────────────────────────────────────────
class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= 60.0:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = 60.0 - (now - self._ts[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] sleeping {sleep_for:.1f}s ...")
                    time.sleep(sleep_for)
            self._ts.append(time.time())

rate_limiter = RateLimiter(CONFIG["max_rpm"])


# ── Per-Question Token Accumulator ─────────────────────────────────────────────
_question_tokens: dict = {}

def _reset_question_tokens() -> None:
    global _question_tokens
    _question_tokens = {}

def _add_tokens(step: str, input_tok: int, output_tok: int) -> None:
    if step not in _question_tokens:
        _question_tokens[step] = {'input': 0, 'output': 0}
    _question_tokens[step]['input']  += int(input_tok  or 0)
    _question_tokens[step]['output'] += int(output_tok or 0)

def _get_question_token_totals() -> tuple:
    total_in  = sum(v['input']  for v in _question_tokens.values())
    total_out = sum(v['output'] for v in _question_tokens.values())
    return total_in, total_out

def _estimate_cost_usd(total_input: int, total_output: int) -> float:
    rate_in  = CONFIG.get('gemini_cost_input_per_1m',  0.10) / 1_000_000
    rate_out = CONFIG.get('gemini_cost_output_per_1m', 0.40) / 1_000_000
    return total_input * rate_in + total_output * rate_out


# ── LLM Call ───────────────────────────────────────────────────────────────────
def call_llm(
    messages: list,
    model: str = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
    json_mode: bool = False,
    step: str = 'generate',
) -> str:
    """Call Gemini with retry. Accumulates tokens to _question_tokens[step]."""
    model = model or LLM_MODEL
    system_instruction = None
    user_parts = []
    for msg in messages:
        if msg["role"] == "system":
            system_instruction = msg["content"]
        else:
            user_parts.append(msg["content"])
    contents = "\n\n".join(user_parts) if user_parts else ""

    cfg_kwargs = dict(temperature=temperature, max_output_tokens=max_tokens)
    if json_mode:
        cfg_kwargs["response_mime_type"] = "application/json"
    if system_instruction:
        cfg_kwargs["system_instruction"] = system_instruction

    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            resp = genai_client.models.generate_content(
                model=model,
                contents=contents,
                config=genai_types.GenerateContentConfig(**cfg_kwargs),
            )
            if step and resp.usage_metadata:
                _add_tokens(step,
                    resp.usage_metadata.prompt_token_count     or 0,
                    resp.usage_metadata.candidates_token_count or 0)
            return resp.text.strip()
        except Exception as e:
            delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1}/{CONFIG['llm_max_retries']} failed ({e}). Retrying in {delay}s...")
            time.sleep(delay)
    raise RuntimeError(f"LLM call failed after {CONFIG['llm_max_retries']} attempts")


# ── Heuristic Scoring Utilities ────────────────────────────────────────────────
def _tokenize(text: str) -> list:
    return _re.sub(r'[^\w\s]', '', text.lower()).split()

def token_f1_score(pred: str, ref: str) -> float:
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks  = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((_Counter(pred_toks) & _Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall    = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)

def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    nums_ref  = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', ref)]
    nums_pred = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', pred)]
    if not nums_ref:
        return None
    hits = sum(
        1 for r in nums_ref
        if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred)
    )
    return hits / len(nums_ref)

def compute_decision_from_text(answer_text: str) -> str:
    lower = answer_text.lower()
    refusal_phrases = [
        'insufficient', 'not contain', 'not available', 'cannot find',
        "don't have", 'no information', 'not provided', 'unable to',
        'not enough', 'not present', 'not found', 'insufficient data',
    ]
    return 'refuse' if any(p in lower for p in refusal_phrases) else 'answer'


# ── Structured Judge (matches CrewAI) ─────────────────────────────────────────
class JudgeOutput(BaseModel):
    score:          float = Field(default=0.0, description='Score 0.0-1.0')
    claims_covered: float = Field(default=0.0, description='Fraction of key facts covered 0.0-1.0')
    reason:         str   = Field(default='',  description='Short explanation')

def _build_correctness_prompt(question: str, reference_answer: str, candidate_answer: str) -> str:
    return (
        'Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. '
        '1 = correct value, correct units, correct period. '
        '0 = wrong number, wrong company, wrong period, or completely missing. '
        'Give partial credit for answers close but with unit errors. '
        'Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Reference answer:\n{reference_answer}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _build_faithfulness_prompt(question: str, context: str, candidate_answer: str) -> str:
    return (
        'Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. '
        '1 = every number and claim is directly supported by the context. '
        '0 = answer contains numbers or claims not present in the context (hallucinated). '
        'Also set claims_covered: fraction of claims in the candidate supported by the context. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Retrieved context:\n{context}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _safe_judge_call(prompt_text: str, task_name: str) -> JudgeOutput:
    _fb = JudgeOutput(score=0.0, claims_covered=0.0, reason='judge fallback — all attempts failed')
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            response = genai_client.models.generate_content(
                model=CONFIG["judge_model"],
                contents=prompt_text,
                config=genai_types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=JudgeOutput,
                    temperature=CONFIG.get('temperature_judge', 0.1),
                ),
            )
            if response.usage_metadata:
                _add_tokens(task_name,
                    response.usage_metadata.prompt_token_count     or 0,
                    response.usage_metadata.candidates_token_count or 0)
            result = response.parsed
            if result is None:
                result = JudgeOutput(**json.loads(response.text))
            return result
        except Exception as e:
            print(f"  [WARN] Judge ({task_name}) attempt {attempt+1} failed: {e}")
            if attempt < CONFIG["llm_max_retries"] - 1:
                time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return _fb

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_correctness_prompt(question, reference_answer, candidate_answer),
        task_name='judge_correctness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_faithfulness_prompt(question, context[:CONFIG.get('max_context_chars', 7000)], candidate_answer),
        task_name='judge_faithfulness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

print("Client, rate limiter, token tracking, heuristics, and judge ready.")

Loading embedding model...
  ok sentence-transformers/all-mpnet-base-v2
  ok Gemini client ready (model: gemini-2.5-flash)
Client, rate limiter, token tracking, heuristics, and judge ready.


## 4. Load ChromaDB Vector Store

In [6]:
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_db_path"])
collections = chroma_client.list_collections()
print(f"  Available collections: {[c.name for c in collections]}")
collection = chroma_client.get_collection(collections[0].name)
print(f"  ok Using '{collections[0].name}'  ({collection.count():,} chunks)")

Connecting to ChromaDB...
  Available collections: ['sec_sections', 'sec_filings_text', 'sec_filings', 'sec_filings_tables']
  ok Using 'sec_sections'  (588 chunks)


## 5. Retrieval — Dense Search Only

In [7]:
# def retrieve(query: str, top_k: int = None) -> list:
#     """
#     Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
#     Returns list of dicts: text, metadata, score, chunk_id.
#     """
#     k = top_k or CONFIG["dense_top_k"]
#     query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
#     results = collection.query(
#         query_embeddings=[query_vec],
#         n_results=min(k, collection.count()),
#         include=["documents", "metadatas", "distances", "ids"],
#     )
#     chunks = []
#     for doc, meta, dist, cid in zip(
#         results["documents"][0],
#         results["metadatas"][0],
#         results["distances"][0],
#         results["ids"][0],
#     ):
#         chunks.append({
#             "text":     doc,
#             "metadata": meta,
#             "score":    float(1.0 - dist),  # cosine distance -> similarity
#             "chunk_id": cid,
#         })
#     return chunks

In [8]:
def retrieve(query: str, top_k: int = None) -> list:
    """
    Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
    Returns list of dicts: text, metadata, score, chunk_id.
    """
    k = top_k or CONFIG["dense_top_k"]
    query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],  
    )

    chunks = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0], 
    ):
        chunks.append({
            "text": doc,
            "metadata": meta,
            "score": float(1.0 - dist),  # cosine distance -> similarity
            "chunk_id": cid,
        })

    return chunks

## 6. Generation

In [9]:
GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers -- include units (millions, billions, %), fiscal periods, and line item names. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)


def format_context(chunks: list, max_chars: int = None) -> str:
    """Format retrieved chunks into a numbered context string."""
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        header = f"[{i}] {m.get('ticker','?')} {m.get('form_type','?')} {m.get('filing_year','?')}"
        parts.append(f"{header}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return context[:max_chars]


def generate(query: str, chunks: list) -> str:
    """Generate an answer from retrieved chunks."""
    context  = format_context(chunks)
    user_msg = f"Question:\n{query}\n\nRetrieved context:\n{context}"
    return call_llm(
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=CONFIG["temperature_generator"],
        max_tokens=CONFIG["max_tokens_answer"],
    )

## 7. Full Baseline RAG Pipeline

In [10]:
def baseline_rag(query: str) -> dict:
    """
    Baseline RAG: Dense Retrieval -> LLM Generation.
    Returns: query, answer, chunks, context.
    """
    chunks  = retrieve(query)
    context = format_context(chunks)
    answer  = generate(query, chunks)

    # print(f'chunks: {chunks}')
    # print('\n\n\---------------')

    # print(f'context: {context}')
    # print('\n\n\---------------')

    # print(f'answer: {answer}')
    return {"query": query, "answer": answer, "chunks": chunks, "context": context}

### Quick Demo

In [11]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
# demo_q = "What was CVX's total net revenue for fiscal year 2023?"

result = baseline_rag(demo_q)

print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Retrieved {len(result['chunks'])} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  score={c['score']:.3f}")

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: The provided context does not contain Apple's total net revenue for fiscal year 2024. The available documents are a 10-Q

Retrieved 5 chunks:
  [1] AAPL    10-Q   year=?  score=0.064
  [2] AAPL    10-Q   year=?  score=0.063
  [3] AAPL    10-K   year=?  score=0.063
  [4] AAPL    10-Q   year=?  score=0.063
  [5] AAPL    10-K   year=?  score=0.062


## 8. Evaluation (LLM-as-Judge)

Uses the same judge prompts as `langgraph_agentic_rag_sec_v2`.
- **Correctness** (0-1): does the answer match the reference on values, units, fiscal period?
- **Faithfulness** (0-1): is every claim grounded in the retrieved context?
- **Claims covered** (0-1): fraction of reference facts present in the candidate.
- **Refusal accuracy**: for adversarial questions, did the model correctly say it cannot answer?

In [12]:
eval_df = pd.read_csv(CONFIG["eval_path"])
eval_df = eval_df[eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

print(f"Evaluation split : '{CONFIG['eval_split']}'  ({len(eval_df)} questions)")
print(eval_df["question_type"].value_counts().to_string())

Evaluation split : 'test'  (80 questions)
question_type
extractive     20
paraphrased    20
reasoning      20
adversarial    20


In [13]:
results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating baseline"):
    question          = str(row["question"])
    reference_answer  = str(row["expected_answer"]).strip()
    question_id       = row.get("question_id", idx)
    question_type     = row.get("question_type", "unknown")
    company           = row.get("company", "")
    ticker            = row.get("ticker", "")
    form_type         = row.get("form_type", "")
    filing_year       = row.get("filing_year", None)
    expected_decision = str(row.get("expected_decision", "answer")).lower().strip()

    print(f'\nQ{idx+1}/{len(eval_df)} [{question_type}]  {ticker}  {filing_year}')

    # ── Reset token accumulator ────────────────────────────────────────────────
    _reset_question_tokens()

    # ── Run pipeline ──────────────────────────────────────────────────────────
    t0 = time.time()
    rag = baseline_rag(question)
    latency_sec  = round(time.time() - t0, 2)
    final_answer = rag["answer"]
    context      = rag["context"]

    # ── Heuristic scores ───────────────────────────────────────────────────────
    t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
    num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
    predicted_decision = compute_decision_from_text(final_answer)
    decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

    # ── LLM Judge ──────────────────────────────────────────────────────────────
    c_score, c_covered, c_reason = llm_judge_correctness(question, reference_answer, final_answer)
    f_score, _,         f_reason = llm_judge_faithfulness(question, context, final_answer)

    # ── Token / cost summary ───────────────────────────────────────────────────
    gen_tok   = _question_tokens.get('generate',          {'input': 0, 'output': 0})
    corr_tok  = _question_tokens.get('judge_correctness', {'input': 0, 'output': 0})
    faith_tok = _question_tokens.get('judge_faithfulness',{'input': 0, 'output': 0})
    total_in, total_out = _get_question_token_totals()
    est_cost = _estimate_cost_usd(total_in, total_out)

    f1_str = f"{t_f1:.2f}" if t_f1 is not None else "N/A"
    print(f'  corr={c_score:.2f}  faith={f_score:.2f}  f1={f1_str}  '
          f'tokens={total_in}in/{total_out}out  est=${est_cost:.5f}')

    results.append({
        # Question metadata
        "question_id":              question_id,
        "question":                 question,
        "question_type":            question_type,
        "company":                  company,
        "ticker":                   ticker,
        "form_type":                form_type,
        "filing_year":              filing_year,
        "reference_answer":         reference_answer,
        "expected_decision":        expected_decision,
        # Answer
        "final_answer":             final_answer,
        "pipeline":                 "baseline_rag",
        # Pipeline metadata
        "n_chunks":                 len(rag["chunks"]),
        "latency_sec":              latency_sec,
        # Heuristic scores
        "token_f1":                 t_f1,
        "numerical_accuracy":       num_acc,
        "decision_accuracy":        decision_accuracy,
        "predicted_decision":       predicted_decision,
        # LLM Judge
        "llm_correctness_score":    c_score,
        "llm_claims_covered":       c_covered,
        "llm_correctness_reason":   c_reason,
        "llm_faithfulness_score":   f_score,
        "llm_faithfulness_reason":  f_reason,
        # Model logging
        "model_generator":          LLM_MODEL,
        "model_judge_correctness":  CONFIG["judge_model"],
        "model_judge_faithfulness": CONFIG["judge_model"],
        # Token & cost
        "tokens_generate_input":    gen_tok["input"],
        "tokens_generate_output":   gen_tok["output"],
        "tokens_judge_corr_input":  corr_tok["input"],
        "tokens_judge_corr_output": corr_tok["output"],
        "tokens_judge_faith_input": faith_tok["input"],
        "tokens_judge_faith_output":faith_tok["output"],
        "tokens_total_input":       total_in,
        "tokens_total_output":      total_out,
        "est_cost_usd":             est_cost,
    })
    time.sleep(60)

results_df = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "baseline_results.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved {len(results_df)} rows -> {out_path}")

Evaluating baseline:   0%|          | 0/80 [00:00<?, ?it/s]


Q1/80 [extractive]  NVDA  2024.0
  corr=0.00  faith=1.00  f1=0.20  tokens=2970in/200out  est=$0.00038


Evaluating baseline:   1%|▏         | 1/80 [01:08<1:30:18, 68.59s/it]


Q2/80 [extractive]  NVDA  2025.0
  corr=0.00  faith=1.00  f1=0.35  tokens=2930in/131out  est=$0.00035


Evaluating baseline:   2%|▎         | 2/80 [02:17<1:29:09, 68.58s/it]

...


Q79/80 [adversarial]  COST  nan
  corr=1.00  faith=1.00  f1=0.00  tokens=3282in/193out  est=$0.00041


Evaluating baseline:  99%|█████████▉| 79/80 [1:31:08<01:08, 68.47s/it]


Q80/80 [adversarial]  COST  nan
  corr=1.00  faith=1.00  f1=0.00  tokens=2850in/150out  est=$0.00035


Evaluating baseline: 100%|██████████| 80/80 [1:32:16<00:00, 69.20s/it]


Saved 80 rows -> results\baseline_results.csv


## 9. Results

In [14]:
print("=" * 60)
print(" BASELINE RAG — EVALUATION RESULTS")
print("=" * 60)

scored_c = results_df[results_df['llm_correctness_score'].notna()]
scored_f = results_df[results_df['llm_faithfulness_score'].notna()]
print(f'\nTotal questions    : {len(results_df)}')
print(f'Correctness judged : {len(scored_c)}')
print(f'Faithfulness judged: {len(scored_f)}')

print('\nOverall metrics:')
for col, label in [
    ('llm_correctness_score',  'LLM Correctness  '),
    ('llm_claims_covered',     'Claims Covered   '),
    ('llm_faithfulness_score', 'LLM Faithfulness '),
    ('token_f1',               'Token F1         '),
    ('numerical_accuracy',     'Numerical Accuracy'),
    ('decision_accuracy',      'Decision Accuracy'),
]:
    vals = results_df[col].dropna()
    if len(vals):
        print(f'  {label}: {vals.mean():.4f}  (n={len(vals)})')

print('\nBy question_type:')
type_cols = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
avail = [c for c in type_cols if c in results_df.columns]
if avail and 'question_type' in results_df.columns:
    print(results_df.groupby('question_type')[avail].mean().round(3).to_string())

if 'latency_sec' in results_df.columns:
    lat = results_df['latency_sec'].dropna()
    if len(lat):
        print(f'\nLatency: mean={lat.mean():.1f}s  median={lat.median():.1f}s  max={lat.max():.1f}s')

print('\nToken & cost summary:')
for col in ['tokens_generate_input', 'tokens_generate_output',
            'tokens_judge_corr_input', 'tokens_judge_corr_output',
            'tokens_judge_faith_input', 'tokens_judge_faith_output',
            'tokens_total_input', 'tokens_total_output']:
    if col in results_df.columns:
        print(f'  {col:35s}: {int(results_df[col].sum()):,}')
if 'est_cost_usd' in results_df.columns:
    print(f'  {"Total estimated cost (USD)":35s}: ${results_df["est_cost_usd"].sum():.4f}')

 BASELINE RAG — EVALUATION RESULTS

Total questions    : 80
Correctness judged : 80
Faithfulness judged: 80

Overall metrics:
  LLM Correctness  : 0.2500  (n=80)
  Claims Covered   : 0.2500  (n=80)
  LLM Faithfulness : 1.0000  (n=80)
  Token F1         : 0.1752  (n=80)
  Numerical Accuracy: 0.2219  (n=54)
  Decision Accuracy: 0.2500  (n=80)

By question_type:
               llm_correctness_score  llm_faithfulness_score  token_f1  decision_accuracy
question_type                                                                            
adversarial                      1.0                     1.0     0.000                1.0
extractive                       0.0                     1.0     0.280                0.0
paraphrased                      0.0                     1.0     0.154                0.0
reasoning                        0.0                     1.0     0.267                0.0

Latency: mean=2.6s  median=2.5s  max=4.5s

Token & cost summary:
  tokens_generate_input         

In [15]:
results_df

,question_id,question,question_type,company,ticker,form_type,filing_year,reference_answer,expected_decision,final_answer,...,model_judge_faithfulness,tokens_generate_input,tokens_generate_output,tokens_judge_corr_input,tokens_judge_corr_output,tokens_judge_faith_input,tokens_judge_faith_output,tokens_total_input,tokens_total_output,est_cost_usd
0,6.0,What was Nvidia's gross margin percentage in f...,extractive,NVIDIA,NVDA,10-K,2024.0,"Nvidia's gross margin was 72.7% in FY2024, up ...",answer,The provided context does not contain informat...,...,gemini-2.5-flash,1330,59,217,61,1423,80,2970,200,0.000377
1,7.0,How many employees did Nvidia report at the en...,extractive,NVIDIA,NVDA,10-K,2025.0,"As of the end of fiscal year 2025, Nvidia had ...",answer,The provided context does not contain informat...,...,gemini-2.5-flash,1331,26,208,52,1391,53,2930,131,0.000345
2,8.0,What was JPMorgan's CET1 capital ratio at the ...,extractive,JPMorgan Chase,JPM,10-K,2023.0,JPMorgan's Standardized and Advanced CET1 capi...,answer,The retrieved context does not contain informa...,...,gemini-2.5-flash,1332,45,212,78,1411,70,2955,193,0.000373
3,9.0,Which federal regulator supervises JPMorgan Ch...,extractive,JPMorgan Chase,JPM,10-K,2024.0,JPMorgan Chase is supervised and regulated by ...,answer,The retrieved context describes NVIDIA's busin...,...,gemini-2.5-flash,1322,31,173,60,1387,57,2882,148,0.000347
4,10.0,What were Bank of America's Total liabilities ...,extractive,Bank of America,BAC,10-Q,2025.0,Bank of America's total liabilities as of Sept...,answer,"I'm sorry, but the provided context does not c...",...,gemini-2.5-flash,1376,43,199,57,1453,70,3028,170,0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,96.0,What internal AI system does Chevron use for d...,adversarial,Chevron,CVX,NaN,NaN,nan,refuse,The provided context does not contain informat...,...,gemini-2.5-flash,1314,41,159,52,1389,70,2862,163,0.000351
76,97.0,What percentage of Walmart employees work remo...,adversarial,Walmart,WMT,NaN,NaN,nan,refuse,"I'm sorry, but the provided context does not c...",...,gemini-2.5-flash,1308,33,149,48,1375,57,2832,138,0.000338
77,98.0,What internal robot efficiency metrics does Wa...,adversarial,Walmart,WMT,NaN,NaN,nan,refuse,The provided context does not contain informat...,...,gemini-2.5-flash,1315,51,170,57,1400,81,2885,189,0.000364
78,99.0,What percentage of Costco warehouses are fully...,adversarial,Costco,COST,NaN,NaN,nan,refuse,The provided context consists of NVIDIA's 10-Q...,...,gemini-2.5-flash,1510,56,172,57,1600,80,3282,193,0.000405
